In [ ]:
from pathlib import Path
import sys

# Locate this package and load the shared v3 LE locations (config/paths.json,
# overridable with the V3LE_ROOT environment variable).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from paths import V3LE_DATA_DIR, V3LE_DIAG_DIR, V3LE_FIG_ROOT, fig_dir, require

In [11]:
import os
import glob
import re
import json
import time
import datetime
import numpy as np
import pandas as pd
import shutil
import psutil
import subprocess

import xarray as xr
from pathlib import Path
from pandas.errors import OutOfBoundsDatetime  # <--- add this

# Optional: graceful xCDAT fallback
try:
    from xcdat import open_dataset as xcdat_open
except Exception:
    def xcdat_open(*args, **kwargs):
        return xr.open_dataset(*args, **kwargs)

In [12]:
def read_exp_dict():    
    run_dict = {
        'v3.LR.piControl.en00': {'name': 'v3.LR.piControl', 'period': '000101-050012'},
    }
    return run_dict 

In [13]:
def read_var_dict():
    var_dict = {
            'SST'     : {'unit': '$\circ$C',     'min': -6,  'max': 6,  'nlev': 11, },
            'SICONC'  : {'unit': '%',            'min': -6,  'max': 6,  'nlev': 11, },
            #'SIMASS'  : {'unit': 'kg m$^{-2}$',  'min': -6,  'max': 6,  'nlev': 11, },
            #'SITHICK' : {'unit': 'm',            'min': -6,  'max': 6,  'nlev': 11, },
            'TS'      : {'unit': 'K',            'min': -6,  'max': 6,  'nlev': 11, },
            'TURFLX'  : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'SHFLX+LHFLX'}, 
            'LHFLX'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, }, 
            'SHFLX'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, }, 
            'FLUT'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLNT'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNT'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNTOA'  : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLNTC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNTC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLUTC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNTOAC' : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'RADNET'  : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FSNS-FLNS'},
            'FSNS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSDS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSUS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FSDS−FSNS'},
            'FSNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSDSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSUSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FSDSC−FSNSC'},
            'FLDS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLNS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLUS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FLDS−FLNS'},
            #'FLUSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'form': 'sigma*TS^4'},
            #'FLDSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FLNSC+FLUSC'},
            'FLNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            #'FLNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FLDSC−FLUSC'},
            #'SWCF_SRF': {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            #'LWCF_SRF': {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            #'ALBEDOC_SRF' : {'unit': '%',        'min': -6,  'max': 6,  'nlev': 13, },
            #'ALBEDO_SRF'  : {'unit': '%',        'min': -6,  'max': 6,  'nlev': 13, },
        }
    return var_dict 

In [14]:
class MonthlyBaseProcessor:
    """
    Build a per-experiment MONTHLY base dataset:
      - fixes time axis and longitude orientation
      - loads requested variables (+ evaluates expressions)
      - computes derived fields (CRE, albedo, etc.)
      - optionally writes masked copies (e.g., ice or land mask)
      - saves one NetCDF per experiment (no seasonal/annual aggregation)

    Later, you can derive seasonal/annual stats from these monthlies quickly.
    """

    def __init__(
        self,
        data_dir,
        group,
        run_dict,
        var_dict,
        results_dir,
        region="global",
        reference_year=1850,      # stored in attrs only (no aggregation here)
        frequency="monthly",
        lon_name="lon",
        lat_name="lat",
        mask_kind="ice",          # "ice", "land", or None
        mask_var="SICONC",
        mask_threshold=15.0,      # % for ice; >0 for land by default
        min_sw=1.0,               # guard for SW albedo divisions
        debug=True,
    ):
        self.data_dir = data_dir
        self.group = group
        self.run_dict = run_dict
        self.var_dict = var_dict
        self.results_dir = results_dir
        self.region = region
        self.reference_year = reference_year
        self.frequency = frequency
        self.lon_name = lon_name
        self.lat_name = lat_name
        self.mask_kind = mask_kind
        self.mask_var = mask_var
        self.mask_threshold = float(mask_threshold)
        self.min_sw = float(min_sw)

        # Regions
        self.region_all = self.get_default_region()
        if region not in self.region_all:
            raise ValueError(f"Unknown region '{region}'. Choose from {list(self.region_all.keys())}.")
        os.makedirs(self.results_dir, exist_ok=True)

        # ---- Performance caches & I/O tuning ----
        self._var_cache  = {}   # (exp, var, period, lat_sel_str, lon_sel_str) -> xr.DataArray
        self._mask_cache = {}   # (exp, period, lat_sel_str, lon_sel_str) -> xr.DataArray(bool)
        self._open_chunks = {"time": 120}   # set None if no dask; or extend with {"lat":90,"lon":180}
        self._io_engine   = "netcdf4"       # or None to let xarray auto-select

        # Default ocean fraction threshold used in SST fallback masking
        self.ocean_frac_thresh = getattr(self, "ocean_frac_thresh", 0.5)
        self.debug = debug

    # ---------- Public API ----------
    def get_default_region(self):
        return {
            "global":   [(-90,  90), (-180, 180)],
            "Atlantic": [(  0,  60), ( 265, 345)],
            "Arctic":   [( 50,  90), (-180, 180)],
            "CONUS":    [( 25,  50), ( 235, 295)],
            "Antarctic":[(-90, -50), (-180, 180)],
            "PolarN":   [( 50,  90), (-180, 180)],
            "Greenland":[( 55,  90), ( -75, -10)],
        }

    def process_all_monthly_bases(self, *, apply_mask=True, filename_tag="monthly_base", compress=True):
        """Build & save the monthly base product for every experiment."""
        paths = {}
        for exp in self.run_dict:
            paths[exp] = self.process_monthly_base(
                exp,
                apply_mask=apply_mask,
                filename_tag=filename_tag,
                compress=compress,
            )
        return paths

    def process_monthly_base(self, exp, *, apply_mask=True, filename_tag="monthly_base", compress=True):
        """Build & save the monthly base product for one experiment."""
        period = self.run_dict[exp]["period"]
        lat_slice, lon_slice = self._region_slices(self.region)

        # --- Build output path & fast skip ---
        out_path = os.path.join(self.results_dir, f"{exp}.{filename_tag}.{period}.nc")
        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            try:
                xr.open_dataset(out_path).close()
                print(f"[SKIP] {exp} monthly base exists -> {out_path}")
                return out_path
            except Exception:
                print(f"[WARN] Existing monthly base not readable; regenerating: {out_path}")

        # 1) Load all raw + expr variables
        ds_all = self._load_all_vars_for_run(exp, period, lat_slice, lon_slice)

        # 2) Derived fields (CRE, albedo, etc.)
        self._compute_derived_fields(ds_all)

        # Assemble dataset
        ds = xr.Dataset(ds_all)

        # 3) Optional mask: persist masked copies with *_imsk suffix
        if apply_mask and self.mask_kind is not None:
            mask = self._build_mask(exp, period, lat_slice, lon_slice)
            if mask.dtype != bool:
                mask = mask.astype(bool)
            # align once against any representative variable with time dim
            for v in list(ds.data_vars):
                dv = ds[v]
                m, dv = xr.align(mask, dv, join="inner") if "time" in dv.dims and "time" in mask.dims else (mask, dv)
                ds[v] = dv
                ds[f"{v}_imsk"] = dv.where(m)

        # 4) Attach helpful attrs
        ds.attrs.update({
            "experiment": exp,
            "period": str(period),
            "region": str(self.region),
            "mask_kind": str(self.mask_kind),
            "mask_var": str(self.mask_var),
            "mask_threshold": str(self.mask_threshold),
            "reference_year": str(self.reference_year),
            "product": "monthly_base",
        })

        # 5) Save with compression + chunking
        if compress:
            encoding = {}
            for vn, da in ds.data_vars.items():
                enc = {"zlib": True, "complevel": 4}
                if "time" in da.dims:
                    enc["chunksizes"] = tuple(12 if d == "time" else min(da.sizes[d], 256) for d in da.dims)
                encoding[vn] = enc
            ds.to_netcdf(out_path, encoding=encoding)
        else:
            ds.to_netcdf(out_path)

        print(f"[INFO] Saved monthly base -> {out_path}")
        return out_path

    # ---------- Helpers: region, lon, time ----------
    def _region_slices(self, regnam):
        (lat0, lat1), (lon0, lon1) = self.region_all[regnam]
        lat_sel = slice(lat0, lat1)
        # longitudes normalized to [-180, 180]; handle wrap-around
        if lon0 <= lon1:
            lon_sel = slice(lon0, lon1)
        else:
            lon_sel = (slice(lon0, 180), slice(-180, lon1))
        return lat_sel, lon_sel

    @staticmethod
    def _normalize_longitude(ds, lon_name="lon"):
        lons = ds[lon_name].values
        if np.min(lons) > -1:
            ds = ds.assign_coords({lon_name: ((lons + 180) % 360 - 180)})
            ds = ds.sortby(lon_name)
        return ds

    @staticmethod
    def _ensure_valid_time(ds, period, frequency="monthly"):
        """
        Reconstruct monthly time coordinate if missing/undecodable, using 'YYYYMM-YYYYMM'.

        For "normal" years (within pandas' datetime64[ns] range),
        use pandas.date_range. For very early/late years (e.g., piControl
        starting at 0001 with a noleap calendar), fall back to
        xarray.cftime_range.
        """
        # 1) If time is already datetime-like or cftime-like, keep it as-is
        try:
            t = ds["time"]
            # Already real datetime64?
            if np.issubdtype(t.dtype, np.datetime64):
                return ds

            # Already cftime objects? (dtype=object but first element is cftime.*)
            if t.size > 0:
                first = t.values[0]
                if "cftime" in type(first).__module__:
                    return ds
        except Exception:
            pass

        # 2) Parse period string: 'YYYYMM-YYYYMM'
        start_str, end_str = period.split("-")
        sy, sm = int(start_str[:4]), int(start_str[4:])
        ey, em = int(end_str[:4]), int(end_str[4:])

        # 3) Try pandas first; if that overflows, switch to cftime
        use_cftime = False
        try:
            start = pd.Timestamp(year=sy, month=sm, day=1)
            end   = pd.Timestamp(year=ey, month=em, day=1)
            new_time = pd.date_range(start=start, end=end, freq="MS")
        except (OutOfBoundsDatetime, OverflowError, ValueError):
            use_cftime = True

        if use_cftime:
            # Calendar: prefer variable attribute, then global, then default to noleap
            cal = getattr(ds.get("time", None), "attrs", {}).get(
                "calendar",
                ds.attrs.get("calendar", "noleap"),
            )

            # xarray uses 'noleap' or '365_day' for no-leap calendars
            new_time = xr.cftime_range(
                start=f"{sy:04d}-{sm:02d}-01",
                end=f"{ey:04d}-{em:02d}-01",
                freq="MS",
                calendar=cal,
            )

        # 4) Sanity check length vs existing time dimension
        nt = ds.sizes.get("time", len(new_time))
        if nt != len(new_time):
            raise ValueError(
                f"[ERROR] Expected {len(new_time)} monthly steps from {period}, "
                f"but dataset has {nt}."
            )

        print(
            f"[INFO] Reconstructing time from {new_time[0]} to {new_time[-1]} "
            f"(calendar={getattr(new_time, 'calendar', ds.attrs.get('calendar', 'standard'))})"
        )
        return ds.assign_coords(time=new_time)

    # ---------- IO ----------
    def _open_subset(self, path, period, lat_slice, lon_slice):
        # Fast open: no decode_times on open; we fix later. Chunk time for responsiveness.
        ds = xr.open_dataset(
            path,
            decode_times=False,
            engine=self._io_engine,
            chunks=self._open_chunks or None,
            cache=False,
        )
        ds = self._ensure_valid_time(ds, period, self.frequency)
        ds = self._normalize_longitude(ds, self.lon_name)

        ds = ds.sel(lat=lat_slice)
        # handle dateline wrap
        if isinstance(lon_slice, tuple):
            part1 = ds.sel(lon=lon_slice[0])
            part2 = ds.sel(lon=lon_slice[1])
            ds = xr.concat([part1, part2], dim="lon")
        else:
            ds = ds.sel(lon=lon_slice)
        return ds

    def _open_var(self, exp, var, period, lat_slice, lon_slice):
        """
        Open a variable with smart fallbacks + unit harmonization (cached).
        - SICONC <- ICEFRAC (fraction→percent)
        - SST    <- TS (K→°C) + mask to ocean
        """
        lat_key = str(lat_slice); lon_key = str(lon_slice)
        cache_key = (exp, var, period, lat_key, lon_key)
        if cache_key in self._var_cache:
            return self._var_cache[cache_key]

        base_path = os.path.join(self.data_dir, self.group, f"{exp}.{var}.{period}.nc")
        print(f"[DIAG]: open file {base_path}")
        if os.path.exists(base_path):
            da = self._open_subset(base_path, period, lat_slice, lon_slice)[var]
            self._var_cache[cache_key] = da
            return da

        # Fallbacks
        if var == "SICONC":
            alt_var = "ICEFRAC"
            alt_path = os.path.join(self.data_dir, self.group, f"{exp}.{alt_var}.{period}.nc")
            if os.path.exists(alt_path):
                ice = self._open_subset(alt_path, period, lat_slice, lon_slice)[alt_var]
                out = (ice * 100.0).rename("SICONC")
                out.attrs = dict(ice.attrs)
                out.attrs["units"] = "%"
                out.attrs["long_name"] = out.attrs.get("long_name", "Sea Ice Concentration")
                self._var_cache[cache_key] = out
                return out
            raise FileNotFoundError(f"Neither SICONC nor ICEFRAC found for {exp} {period}")

        if var == "SST":
            alt_var = "TS"
            alt_path = os.path.join(self.data_dir, self.group, f"{exp}.{alt_var}.{period}.nc")
            if os.path.exists(alt_path):
                ts = self._open_subset(alt_path, period, lat_slice, lon_slice)[alt_var]
                sst = (ts - 273.15).rename("SST")
                sst.attrs = dict(ts.attrs)
                sst.attrs["units"] = "degC"
                sst.attrs["long_name"] = sst.attrs.get("long_name", "Sea Surface Temperature (derived from TS)")
                ocean_mask = self._get_ocean_mask(exp, period, lat_slice, lon_slice)
                ocean_mask, sst = xr.align(ocean_mask.astype(bool), sst, join="inner")
                sst = sst.where(ocean_mask)
                self._var_cache[cache_key] = sst
                return sst
            raise FileNotFoundError(f"Neither SST nor TS found for {exp} {period}")

        raise FileNotFoundError(f"File for {var} not found: {base_path}")

    def _get_ocean_mask(self, exp, period, lat_slice, lon_slice):
        """
        Return boolean mask (True = ocean) using available fraction fields (cached).
        Tries: OCNFRAC (0..1), LANDFRAC (0..1), SFTLF (0..100).
        """
        lat_key = str(lat_slice); lon_key = str(lon_slice)
        key = (exp, period, lat_key, lon_key)
        if key in self._mask_cache:
            return self._mask_cache[key]

        def _try(vnam):
            pth = os.path.join(self.data_dir, self.group, f"{exp}.{vnam}.{period}.nc")
            if os.path.exists(pth):
                return self._open_subset(pth, period, lat_slice, lon_slice)[vnam]
            return None

        thr = self.ocean_frac_thresh

        ocnfrac = _try("OCNFRAC")
        if ocnfrac is not None:
            mask = (ocnfrac > thr)
        else:
            landfrac = _try("LANDFRAC")
            if landfrac is not None:
                mask = ((1.0 - landfrac) > thr)
            else:
                sftlf = _try("SFTLF")  # percent land
                if sftlf is not None:
                    ocnfrac_guess = 1.0 - (sftlf / 100.0)
                    mask = (ocnfrac_guess > thr)
                else:
                    raise FileNotFoundError("Cannot construct ocean mask: missing OCNFRAC, LANDFRAC, and SFTLF.")

        mask = mask.astype(bool)
        self._mask_cache[key] = mask
        return mask

    # ---------- Variable loading ----------
    def _eval_expr(self, exp, expr, period, lat_slice, lon_slice):
        """Safely evaluate a simple algebraic expression of already-available variables."""
        expr = expr.replace("−", "-")
        tokens = [v.strip() for v in expr.replace("+", " ").replace("-", " ").split()]
        scope = {}
        for v in tokens:
            if v and v not in scope:
                scope[v] = self._open_var(exp, v, period, lat_slice, lon_slice)
        return eval(expr, {"__builtins__": {}}, scope)

    def _load_all_vars_for_run(self, exp, period, lat_slice, lon_slice):
        ds_all = {}
        for vname, meta in self.var_dict.items():
            vexpr = meta.get("expr")
            if vexpr is None:
                da = self._open_var(exp, vname, period, lat_slice, lon_slice)
            else:
                da = self._eval_expr(exp, vexpr, period, lat_slice, lon_slice)
            ds_all[vname] = da
            # Lightweight debug (avoid .values which loads data)
            try:
                print(f"[DEBUG] {exp}:{vname} dims={dict(da.sizes)} time_len={da.sizes.get('time', 0)}")
            except Exception:
                pass
        return ds_all

    # ---------- Derived fields ----------
    def _compute_derived_fields(self, ds_all):
        sigma = 5.67e-8

        # Upwelling LW (clear) from TS
        if "TS" in ds_all and "FLUSC" not in ds_all:
            ds_all["FLUSC"] = sigma * ds_all["TS"] ** 4
            ds_all["FLUSC"].attrs.update({"long_name": "Surface upwelling LW (clear-sky)", "units": "W m-2"})

        # Downwelling LW (clear) if missing
        if "FLNSC" in ds_all and "FLUSC" in ds_all and "FLDSC" not in ds_all:
            ds_all["FLDSC"] = ds_all["FLNSC"] + ds_all["FLUSC"]
            ds_all["FLDSC"].attrs.update({"long_name": "Surface downwelling LW (clear-sky)", "units": "W m-2"})

        # Surface CRE components (all-sky vs clear-sky)
        if "FSNS" in ds_all and "FSNSC" in ds_all:
            ds_all["SWCF_SRF"] = ds_all["FSNS"] - ds_all["FSNSC"]
            ds_all["SWCF_SRF"].attrs.update({"long_name": "Surface SW cloud radiative effect", "units": "W m-2"})

        if "FLNS" in ds_all and "FLNSC" in ds_all:
            ds_all["LWCF_SRF"] = ds_all["FLNS"] - ds_all["FLNSC"]
            ds_all["LWCF_SRF"].attrs.update({"long_name": "Surface LW cloud radiative effect", "units": "W m-2"})

        if "SWCF_SRF" in ds_all and "LWCF_SRF" in ds_all:
            ds_all["CRE_SRF"] = ds_all["SWCF_SRF"] + ds_all["LWCF_SRF"]
            ds_all["CRE_SRF"].attrs.update({"long_name": "Surface cloud radiative effect", "units": "W m-2"})

        # TOA CRE
        if "FSNT" in ds_all and "FSNTC" in ds_all:
            ds_all["SWCF"] = ds_all["FSNT"]  - ds_all["FSNTC"]
            ds_all["SWCF"].attrs.update({"long_name": "TOA SW cloud radiative effect", "units": "W m-2"})

        if "FLNT" in ds_all and "FLNTC" in ds_all:
            ds_all["LWCF"] = ds_all["FLNTC"] - ds_all["FLNT"]
            ds_all["LWCF"].attrs.update({"long_name": "TOA LW cloud radiative effect", "units": "W m-2"})

        if "SWCF" in ds_all and "LWCF" in ds_all:
            ds_all["CRE"] = ds_all["SWCF"] + ds_all["LWCF"]
            ds_all["CRE"].attrs.update({"long_name": "TOA cloud radiative effect", "units": "W m-2"})

        # Surface albedo (all-sky & clear-sky). Also expose FSUS/FSUSC if missing.
        if "FSDS" in ds_all and "FSNS" in ds_all:
            ds_all.setdefault("FSUS", ds_all["FSDS"] - ds_all["FSNS"])
            alb = 1.0 - (ds_all["FSNS"] / ds_all["FSDS"]).where(ds_all["FSDS"] > self.min_sw)
            ds_all["ALBEDO_SRF"] = alb.clip(0.0, 1.0)
            ds_all["ALBEDO_SRF"].attrs.update({"long_name": "Surface albedo (all-sky)", "units": "1"})

        if "FSDSC" in ds_all and "FSNSC" in ds_all:
            ds_all.setdefault("FSUSC", ds_all["FSDSC"] - ds_all["FSNSC"])
            alb_c = 1.0 - (ds_all["FSNSC"] / ds_all["FSDSC"]).where(ds_all["FSDSC"] > self.min_sw)
            ds_all["ALBEDOC_SRF"] = alb_c.clip(0.0, 1.0)
            ds_all["ALBEDOC_SRF"].attrs.update({"long_name": "Surface albedo (clear-sky)", "units": "1"})

    # ---------- Masks ----------
    def _build_mask(self, exp, period, lat_slice, lon_slice):
        if self.mask_kind is None:
            return None
        da = self._open_var(exp, self.mask_var, period, lat_slice, lon_slice)
        if self.mask_kind in ("ice", "land"):
            return xr.where(da > self.mask_threshold, True, False)
        raise ValueError("mask_kind must be 'ice', 'land', or None.")
        
    def _monthwise_trend(self, da, t_year=None, tdim=None, scale=None):
        """
        Compute month-wise linear trend grouped by calendar month.

        For each calendar month m (1..12), regress the variable against a
        year index:

            slope_m = cov(year_index, X) / var(year_index)

        Output dims: (month, <other spatial dims>).
        Debug mode prints helpful info including t_year and month ranges.
        """
        if tdim not in da.dims:
            return None

        # ---- Build / use t_year for debug only ----
        if t_year is None:
            idx = xr.DataArray(
                np.arange(da.sizes[tdim], dtype="float64"),
                dims=(tdim,),
                coords={tdim: da[tdim]},
                name="time_index",
            )
            t_year = idx / 12.0

        # ============================
        # Debug: Print t_year & month info
        # ============================
        if self.debug:
            print(f"[DEBUG] ---- Trend Debug for {da.name} ----")
            print(f"[DEBUG] Data dims: {dict(da.sizes)}  time_len={da.sizes[tdim]}")

            # Print t_year sample & min/max
            try:
                print(
                    f"[DEBUG] t_year: min={float(t_year.min().values):.3f}, "
                    f"max={float(t_year.max().values):.3f}"
                )
                print(f"[DEBUG] t_year first 5: {t_year.isel({tdim: slice(0, 5)}).values}")
            except Exception:
                print("[DEBUG] Could not compute t_year min/max or sample")

        # ---- Extract months & years from the time coordinate ----
        try:
            months = da[tdim].dt.month
            years  = da[tdim].dt.year
        except Exception as e:
            raise ValueError(
                f"Time coordinate along '{tdim}' is not datetime-like; "
                f"cannot use .dt.month/.dt.year: {e}"
            )

        # Year index (0, 1, 2, ...) used in regression
        year0 = int(years.min().values)
        year_index = (years - year0).astype("float64")

        if self.debug:
            uniq_months = np.unique(months.values)
            print(f"[DEBUG] Detected months in series: {uniq_months}")

        # ============================
        # Month-by-month regression
        # ============================
        out_list = []
        for m in range(1, 13):
            mask = (months == m)
            if mask.sum() == 0:
                # No data for this month (e.g., truncated period)
                continue

            # Subset along time for this month only
            y = da.where(mask, drop=True)
            t = year_index.where(mask, drop=True)

            # Demean along time
            y0 = y - y.mean(dim=tdim)
            t0 = t - t.mean(dim=tdim)

            num = (t0 * y0).sum(dim=tdim)
            den = (t0 ** 2).sum(dim=tdim)
            slope_m = num / den

            if scale == "per_100yr":
                slope_m = slope_m * 100.0

            # Attach month coordinate
            slope_m = slope_m.expand_dims(month=[m])
            out_list.append(slope_m)

        if not out_list:
            # No months had data
            return None

        slope = xr.concat(out_list, dim="month")

        # Debug: slope dims
        if self.debug:
            print(f"[DEBUG] slope dims for {da.name}: {dict(slope.sizes)}")
            try:
                print(f"[DEBUG] slope.month values: {slope['month'].values}")
            except Exception:
                print("[DEBUG] slope has no explicit month coordinate")

        return slope

    def save_linear_trend(
        self,
        exp,
        base_filename_tag="monthly_base",
        trend_filename_tag="trend_500yr_monthly",
        scale="per_year",  # or 'per_100yr'
    ):
        """
        Compute month-by-month linear trends at each grid point for all
        time-dependent variables in the monthly_base file, and save to
        <exp>.<trend_filename_tag>.<period>.nc.

        The trend is computed separately for each calendar month using a
        simple linear regression along the time dimension:

            slope = cov(t, X) / var(t)

        where t is an index-based time in years (0, 1/12, 2/12, ...).

        Output variables are named <var>_trend_month(month, lat, lon) and
        have units per year (or per 100 years if scale='per_100yr').
        """
        period = self.run_dict[exp]["period"]
        in_path = os.path.join(self.results_dir, f"{exp}.{base_filename_tag}.{period}.nc")
        if not os.path.exists(in_path):
            raise FileNotFoundError(f"Base monthly file not found: {in_path}")

        out_path = os.path.join(self.results_dir, f"{exp}.{trend_filename_tag}.{period}.nc")
        print(f"[INFO] Opening monthly base for month-by-month trend: {in_path}")

        ds = xr.open_dataset(in_path)

        # ---- detect time dimension ----
        if "time" in ds.coords or "time" in ds.dims:
            tdim = "time"
        elif "year" in ds.coords or "year" in ds.dims:
            tdim = "year"
        else:
            raise ValueError("No 'time' or 'year' dimension found in monthly base file.")

        t = ds[tdim]
        nt = t.size
        if nt % 12 != 0:
            raise ValueError(f"Time dimension length {nt} is not a multiple of 12; "
                             "cannot form full calendar months for trend.")

        # ---- build an index-based time coordinate in years ----
        # e.g., 0, 1/12, 2/12, ..., (nt-1)/12
        idx = xr.DataArray(
            np.arange(nt, dtype="float64"),
            dims=(tdim,),
            coords={tdim: t},
            name="time_index",
        )
        t_year = idx / 12.0  # units: years since start
                
        trend_vars = {}
        for vname, da in ds.data_vars.items():
            if tdim not in da.dims:
                # skip non-time-dependent vars
                continue

            print(f"[INFO] Computing month-by-month trend for {vname}")
            try:
                slope = self._monthwise_trend(da, t_year=t_year, tdim=tdim, scale=scale)

            except Exception as e:
                print(f"[WARN] month-wise trend failed for {vname}: {e}")
                continue

            if slope is None:
                continue

            slope_name = f"{vname}_trend_month"
            slope = slope.rename(slope_name)

            # copy basic attrs
            attrs = dict(da.attrs)
            base_long = attrs.get("long_name", vname)
            if scale == "per_100yr":
                attrs["long_name"] = f"Linear trend of {base_long} by month (per 100 years)"
                attrs["units"] = attrs.get("units", "") + " / 100 yr"
            else:
                attrs["long_name"] = f"Linear trend of {base_long} by month (per year)"
                attrs["units"] = attrs.get("units", "") + " / yr"

            slope.attrs = attrs
            trend_vars[slope_name] = slope

            print(
                f"[INFO]  -> {slope_name} dims={dict(slope.sizes)} "
                f"(month={slope.sizes.get('month', 'NA')})"
            )

        if not trend_vars:
            raise RuntimeError("No monthly trends were computed; check input dataset.")

        ds_trend = xr.Dataset(trend_vars)

        # carry over lat/lon coords if present
        for coord in (self.lat_name, self.lon_name):
            if coord in ds.coords:
                ds_trend = ds_trend.assign_coords({coord: ds[coord]})

        # global attrs for bookkeeping
        ds_trend.attrs.update(
            {
                "experiment": exp,
                "period": str(period),
                "source_monthly_base": os.path.basename(in_path),
                "trend_scale": scale,
                "description": (
                    "Month-by-month linear trends from piControl monthly_base, "
                    "estimated via index-based regression along time."
                ),
            }
        )

        # basic compression
        encoding = {}
        for vn, da in ds_trend.data_vars.items():
            encoding[vn] = {"zlib": True, "complevel": 4}

        ds_trend.to_netcdf(out_path, encoding=encoding)
        print(f"[INFO] Saved monthly trend file -> {out_path}")
        return out_path
    

In [15]:
if __name__ == "__main__":
    # --- Paths
    data_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/CVDP_RGD"
    out_dir  = f"{V3LE_DATA_DIR}"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # --- Config
    mask_kind      = "ice"
    mask_var       = "SICONC"
    mask_threshold = 15.0
    new_period     = "000101-050012"
    regnam         = "global"
    group          = "v3.LR.piControl"
    ref_year       = 1

    base_filename_tag  = "monthly_base"     # must match what you used earlier
    trend_filename_tag = "trend_500yr_monthly"  # recommended name
    scale              = "per_year"         # or "per_100yr"

    # --- Read experiment and variable configs
    var_dict = read_var_dict()
    run_dict = read_exp_dict()

    # Force period to match your piControl files
    for key in run_dict:
        run_dict[key]["period"] = new_period

    # Output directory (same as monthly_base directory)
    results_dir = os.path.join(out_dir, regnam)
    Path(results_dir).mkdir(parents=True, exist_ok=True)

    # --- Build processor
    processor = MonthlyBaseProcessor(
        data_dir=data_dir,
        group=group,
        run_dict=run_dict,
        var_dict=var_dict,
        results_dir=results_dir,
        region=regnam,
        reference_year=ref_year,
        mask_kind=mask_kind,
        mask_var=mask_var,
        mask_threshold=mask_threshold,
    )

    # --- Only compute monthly trends (monthly_base already exists!)
    trend_paths = {}
    for exp in run_dict:
        print(f"\n===== Computing monthly trends for experiment: {exp} =====")

        trend_path = processor.save_linear_trend(
            exp,
            base_filename_tag=base_filename_tag,
            trend_filename_tag=trend_filename_tag,
            scale=scale,
        )

        print(f"[INFO] Trend file for {exp}: {trend_path}")
        trend_paths[exp] = trend_path

    print("\nAll trend paths:")
    for exp, p in trend_paths.items():
        print(exp, "->", p)



===== Computing monthly trends for experiment: v3.LR.piControl.en00 =====
[INFO] Opening monthly base for month-by-month trend: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global/v3.LR.piControl.en00.monthly_base.000101-050012.nc
[INFO] Computing month-by-month trend for SST
[DEBUG] ---- Trend Debug for SST ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: min=0.000, max=499.917
[DEBUG] t_year first 5: [0.         0.08333333 0.16666667 0.25       0.33333333]
[DEBUG] Detected months in series: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[DEBUG] slope dims for SST: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> SST_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for SICONC
[DEBUG] ---- Trend Debug for SICONC ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: m

[DEBUG] slope dims for RADNET: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> RADNET_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for FSNS
[DEBUG] ---- Trend Debug for FSNS ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: min=0.000, max=499.917
[DEBUG] t_year first 5: [0.         0.08333333 0.16666667 0.25       0.33333333]
[DEBUG] Detected months in series: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[DEBUG] slope dims for FSNS: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> FSNS_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for FSDS
[DEBUG] ---- Trend Debug for FSDS ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: min=0.000, max=499.917
[DEBUG] t_year f

[DEBUG] slope dims for CRE_SRF: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> CRE_SRF_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for SWCF
[DEBUG] ---- Trend Debug for SWCF ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: min=0.000, max=499.917
[DEBUG] t_year first 5: [0.         0.08333333 0.16666667 0.25       0.33333333]
[DEBUG] Detected months in series: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[DEBUG] slope dims for SWCF: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> SWCF_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for LWCF
[DEBUG] ---- Trend Debug for LWCF ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: min=0.000, max=499.917
[DEBUG] t_year

[DEBUG] slope dims for FSNTOA_imsk: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> FSNTOA_imsk_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for FLNTC_imsk
[DEBUG] ---- Trend Debug for FLNTC_imsk ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: min=0.000, max=499.917
[DEBUG] t_year first 5: [0.         0.08333333 0.16666667 0.25       0.33333333]
[DEBUG] Detected months in series: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[DEBUG] slope dims for FLNTC_imsk: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> FLNTC_imsk_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for FSNTC_imsk
[DEBUG] ---- Trend Debug for FSNTC_imsk ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t

[DEBUG] slope dims for FLUS_imsk: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> FLUS_imsk_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for FLNSC_imsk
[DEBUG] ---- Trend Debug for FLNSC_imsk ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_year: min=0.000, max=499.917
[DEBUG] t_year first 5: [0.         0.08333333 0.16666667 0.25       0.33333333]
[DEBUG] Detected months in series: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[DEBUG] slope dims for FLNSC_imsk: {'month': 12, 'lat': 180, 'lon': 360}
[DEBUG] slope.month values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
[INFO]  -> FLNSC_imsk_trend_month dims={'month': 12, 'lat': 180, 'lon': 360} (month=12)
[INFO] Computing month-by-month trend for FLUSC_imsk
[DEBUG] ---- Trend Debug for FLUSC_imsk ----
[DEBUG] Data dims: {'time': 6000, 'lat': 180, 'lon': 360}  time_len=6000
[DEBUG] t_yea